# Explore Parasol Insurance Metrics

This notebook queries Prometheus for container metrics from the `parasol-insurance`
namespace and displays them as DataFrames. Use it to understand what "normal" looks
like before injecting faults in Section 7.4.

**Prerequisites**: The KubeHeal integration overlay must be deployed (`oc apply -k bootstrap/kubeheal/`).

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta

PROMETHEUS_URL = os.environ.get(
    'PROMETHEUS_URL',
    'http://prometheus-k8s.openshift-monitoring.svc:9090',
)
NAMESPACE = 'parasol-insurance'

def query_prometheus(query: str, hours: int = 1, step: str = '1m') -> pd.DataFrame:
    """Query Prometheus range API and return a tidy DataFrame."""
    end = datetime.utcnow()
    start = end - timedelta(hours=hours)
    params = {
        'query': query,
        'start': start.isoformat() + 'Z',
        'end': end.isoformat() + 'Z',
        'step': step,
    }
    try:
        resp = requests.get(f'{PROMETHEUS_URL}/api/v1/query_range', params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        if data['status'] != 'success' or not data['data']['result']:
            print(f'No data for: {query}')
            return pd.DataFrame()
        rows = []
        for series in data['data']['result']:
            container = series['metric'].get('container', 'unknown')
            for ts, val in series['values']:
                rows.append({
                    'timestamp': datetime.utcfromtimestamp(float(ts)),
                    'container': container,
                    'value': float(val),
                })
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'Query failed: {e}')
        return pd.DataFrame()

print(f'Prometheus: {PROMETHEUS_URL}')
print(f'Namespace:  {NAMESPACE}')

## Memory Usage

Query the working set memory for all containers in the `parasol-insurance` namespace.

In [ ]:
memory_df = query_prometheus(
    f'container_memory_working_set_bytes{{namespace="{NAMESPACE}", container!=""}}'
)

if not memory_df.empty:
    memory_df['value_mb'] = memory_df['value'] / (1024 * 1024)
    print('Memory Usage (MB) per container:')
    display(memory_df.groupby('container')['value_mb'].describe().round(1))
else:
    print('No memory data available. Is the app deployed and ServiceMonitor active?')

## CPU Usage

Query the 5-minute CPU usage rate.

In [ ]:
cpu_df = query_prometheus(
    f'rate(container_cpu_usage_seconds_total{{namespace="{NAMESPACE}", container!=""}}[5m])'
)

if not cpu_df.empty:
    cpu_df['value_millicores'] = cpu_df['value'] * 1000
    print('CPU Usage (millicores) per container:')
    display(cpu_df.groupby('container')['value_millicores'].describe().round(1))
else:
    print('No CPU data available.')

## Summary

This is the baseline. The values above represent what the Parasol Insurance
application looks like under **normal** operation.

In Section 7.4, you will inject faults that push these metrics far outside
these ranges, triggering the self-healing platform's alerts and remediation.